<center><h1>Case_John_HW7</h1></center>

Name: John Case
<br>
Github Username: johncase17
<br>
USC ID: 2249212508

## 1. Multi-class and Multi-Label Classification Using Support Vector Machines

Import packages

In [71]:
%%capture
%pip install -r ../requirements.txt
from sklearn.model_selection import GridSearchCV, train_test_split
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from imblearn.pipeline import Pipeline as IMBPipeline
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import calinski_harabasz_score
from sklearn.cluster import KMeans
from sklearn.svm import SVC, LinearSVC
from sklearn.pipeline import make_pipeline
from imblearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from imblearn.over_sampling import SMOTE

### (a) Download the Anuran Calls (MFCCs) Data Set

In [17]:
path = '../data/Frogs_MFCCs.csv'
raw_data = pd.read_csv(path).drop(columns='RecordID')
y_raw = raw_data[['Family', 'Genus', 'Species']]
x_raw = raw_data.drop(columns=['Family', 'Genus', 'Species'])
x_train, x_test, y_train, y_test = train_test_split(x_raw, y_raw, stratify=y_raw, test_size=0.3, random_state=42)

### (b) Train a classifier for each label

#### (i) Research

Exact match evaluation, as the name suggests, requires all labels to be correct for a given instance, while hamming score compares the average label accuracy per instance. In the context of this exercise, the labels exist in a hierarchy, where species is the most specific label, genus is a set of species, and family is a set of genera.

#### (ii) Train a SVM for each of the labels

In [34]:
train_family = y_train['Family']
test_family = y_test['Family']
train_genus = y_train['Genus']
test_genus = y_test['Genus']
train_specs = y_train['Species']
test_specs = y_test['Species']
def fitSVM(training_x, training_y):
    k_fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    estim = SVC(C=1, kernel='rbf', gamma='scale', random_state=42)
    pipeline = make_pipeline(StandardScaler(), estim)
    parameters = {"svc__C": np.logspace(-2, 3, 10)}
    SVM_CV = GridSearchCV(cv=k_fold, param_grid=parameters, scoring='accuracy', n_jobs=-1, estimator=pipeline)
    return SVM_CV.fit(training_x, training_y)
cv_family = fitSVM(x_train, train_family)
cv_genus = fitSVM(x_train, train_genus)
cv_specs = fitSVM(x_train, train_specs)

In [57]:
k_fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
estim = SVC(C=1, kernel='rbf', random_state=42)
pipeline = make_pipeline(StandardScaler(), estim)
parameters = {"svc__C": np.logspace(-3, 2, 10), "svc__gamma": np.logspace(-3, 2, 10)}
SVM_CV = GridSearchCV(cv=k_fold, param_grid=parameters, scoring='accuracy', n_jobs=-1, estimator=pipeline)
print(estim.get_params())

{'C': 1, 'break_ties': False, 'cache_size': 200, 'class_weight': None, 'coef0': 0.0, 'decision_function_shape': 'ovr', 'degree': 3, 'gamma': 'scale', 'kernel': 'rbf', 'max_iter': -1, 'probability': 'deprecated', 'random_state': 42, 'shrinking': True, 'tol': 0.001, 'verbose': False}


In [62]:
train_family = y_train['Family']
test_family = y_test['Family']
train_genus = y_train['Genus']
test_genus = y_test['Genus']
train_specs = y_train['Species']
test_specs = y_test['Species']
def fitSVM(training_x, training_y):
    k_fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    estim = SVC(C=1, kernel='rbf', gamma='scale', random_state=42)
    pipeline = make_pipeline(StandardScaler(), estim)
    parameters = {"svc__C": np.logspace(-2, 3, 10), "svc__gamma": np.logspace(-3, 1, 10)}
    SVM_CV = GridSearchCV(cv=k_fold, param_grid=parameters, scoring='accuracy', n_jobs=-1, estimator=pipeline)
    return SVM_CV.fit(training_x, training_y)
cv_family = fitSVM(x_train, train_family)
cv_genus = fitSVM(x_train, train_genus)
cv_specs = fitSVM(x_train, train_specs)

In [63]:
family_pred = cv_family.predict(x_test)
genus_pred = cv_genus.predict(x_test)
spec_pred = cv_specs.predict(x_test)
combined_pred = pd.DataFrame({'Family': family_pred, 'Genus': genus_pred, 'Species': spec_pred}).to_numpy()
truth_labels = y_test.to_numpy()
compare = (truth_labels == combined_pred)
hamming_score = compare.mean(axis=1).mean()
exact_score = compare.all(axis=1).mean()
print("Exact match score: " + str(exact_score))
print("Hamming score: " + str(hamming_score))

Exact match score: 0.9856415006947661
Hamming score: 0.9890381349390148


#### (iii) Repeat 1(b)ii with L1-penalized SVMs

In [53]:
def l1SVM(training_x, training_y):
    k_fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    estim = LinearSVC(C=1, penalty='l1', dual='auto', max_iter=10000, random_state=42)
    pipeline = make_pipeline(StandardScaler(), estim)
    parameters = {"linearsvc__C": np.logspace(-3, 2, 10)}
    SVM_CV = GridSearchCV(cv=k_fold, param_grid=parameters, scoring='accuracy', n_jobs=-1, estimator=pipeline)
    return SVM_CV.fit(training_x, training_y)
l1_family = l1SVM(x_train, train_family)
l1_genus = l1SVM(x_train, train_genus)
l1_specs = l1SVM(x_train, train_specs)

In [58]:
family_pred = l1_family.predict(x_test)
genus_pred = l1_genus.predict(x_test)
spec_pred = l1_specs.predict(x_test)
combined_l1_pred = pd.DataFrame({'Family': family_pred, 'Genus': genus_pred, 'Species': spec_pred}).to_numpy()
truth_labels = y_test.to_numpy()
compare_l1 = (truth_labels == combined_l1_pred)
hamming_l1 = compare_l1.mean(axis=1).mean()
exact_l1 = compare_l1.all(axis=1).mean()
print(l1_family.best_params_)
print("L1-penalized Exact match score: " + str(exact_l1))
print("L1-penalized Hamming score: " + str(hamming_l1))

{'linearsvc__C': np.float64(2.1544346900318843)}
L1-penalized Exact match score: 0.9133858267716536
L1-penalized Hamming score: 0.9495136637332098


#### (iv) Repeat 1(b)iii by using SMOTE or any other method for imbalance

In [68]:
def fitSMOTE(training_x, training_y):
    estim = LinearSVC(C=1, penalty='l1', dual='auto', max_iter=10000, random_state=42)
    pipeline = Pipeline([("scaler", StandardScaler()), ("Smote", SMOTE(random_state=42)), ("estimator", estim)])
    k_fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    parameters = {"estimator__C": np.logspace(-3, 2, 10)}
    SVM_CV = GridSearchCV(cv=k_fold, param_grid=parameters, scoring='accuracy', n_jobs=-1, estimator=pipeline)
    return SVM_CV.fit(training_x, training_y)
smote_family = fitSMOTE(x_train, train_family)
smote_genus = fitSMOTE(x_train, train_genus)
smote_specs = fitSMOTE(x_train, train_specs)

In [70]:
family_pred = smote_family.predict(x_test)
genus_pred = smote_genus.predict(x_test)
spec_pred = smote_specs.predict(x_test)
combined_smote_pred = pd.DataFrame({'Family': family_pred, 'Genus': genus_pred, 'Species': spec_pred}).to_numpy()
truth_labels = y_test.to_numpy()
compare_smote = (truth_labels == combined_smote_pred)
hamming_smote = compare_smote.mean(axis=1).mean()
exact_smote = compare_smote.all(axis=1).mean()
print("smote-penalized Exact match score: " + str(exact_smote))
print("smote-penalized Hamming score: " + str(hamming_smote))

smote-penalized Exact match score: 0.8568781843446039
smote-penalized Hamming score: 0.9286706808707735


## 2. K-Means Clustering on a Multi-Class and Multi-Label Data Set

### (a) Use k-means clustering

In [87]:
#basic for loop but k=2 should always have the highest CH-score
def best_k():
    best_k = 1
    best_score = 0
    for k in range(2, 50):
        kmeans = KMeans(n_clusters=k)
        labels = kmeans.fit_predict(x_raw)
        score = calinski_harabasz_score(x_raw, labels)
        if(score > best_score):
            best_score = score
            best_k = k
    return best_k
k= best_k()
k_best = KMeans(n_clusters=k)
labels = k_best.fit_predict(x_raw)
print('best k = ' + str(k))

best k = 2


### (b) Determine which family is the majority

In [115]:
output = y_raw.copy()
cluster_data = pd.Series(labels, index=y_raw.index, name='cluster')
output['cluster'] = cluster_data
cluster_0 = output[output['cluster']==0]
cluster_1 = output[output['cluster']==1]

family0 = cluster_0['Family'].mode().iloc[0]
genus0 = cluster_0['Genus'].mode().iloc[0]
specs0 = cluster_0['Species'].mode().iloc[0]

family1 = cluster_1['Family'].mode().iloc[0]
genus1 = cluster_1['Genus'].mode().iloc[0]
specs1 = cluster_1['Species'].mode().iloc[0]

print("cluster 0 Family: " + str(family0))
print("cluster 0 Genus: " + str(genus0))
print("cluster 0 Species: " + str(specs0))

print("cluster 1 Family: " + str(family1))
print("cluster 1 Genus: " + str(genus1))
print("cluster 1 Species: " + str(specs0))

cluster 0 Family: Leptodactylidae
cluster 0 Genus: Adenomera
cluster 0 Species: AdenomeraHylaedactylus
cluster 1 Family: Hylidae
cluster 1 Genus: Hypsiboas
cluster 1 Species: AdenomeraHylaedactylus


### (c) Calculate the average Hamming distance, Hamming score, and Hamming loss

In [120]:
labels0 = {'Family': family0, 'Genus': genus0, 'Species': specs0, 'cluster': 0}
labels1 = {'Family': family1, 'Genus': genus1, 'Species': specs1, 'cluster': 1}


def compare_rows(row1, row2):
    score = 0
    if(row1['Family'] != row2['Family']):
        score = score + 1
    if(row1['Genus'] != row2['Genus']):
        score = score + 1
    if(row1['Species'] != row2['Species']):
        score = score + 1
    return score

correct = 0
incorrect = 0

for i in range(0, len(output)):
    row = output.iloc[i]
    score = 0
    if(row['cluster'] == 0):
        score = compare_rows(row, labels0)
    else:
        score = compare_rows(row, labels1)
    incorrect = incorrect + score
    correct = 3 - score + correct

In [123]:
h_score = correct/(correct + incorrect)
h_loss = incorrect/(correct + incorrect)
avg_dist = (3*incorrect)/(correct + incorrect)

print('Average Hamming Score: ' + str(h_score))
print('Hamming Loss: ' + str(h_loss))
print('Hamming Score: ' + str(avg_dist))

Average Hamming Score: 0.7014593467685893
Hamming Loss: 0.2985406532314107
Hamming Score: 0.8956219596942321


## 3. ISLR 12.6.2

<img src="../data/12.6.2.jpg" alt="Alt text" width="400" height="300">